In [33]:
import trimesh
import numpy as np

#  LOAD MESH

mesh_path = "/Users/chaitanyaattanti/Downloads/MIXAR/8samples/cylinder.obj"
mesh = trimesh.load(mesh_path, process=False)

vertices = mesh.vertices
faces = mesh.faces

print("Mesh loaded.")
print("Vertices:", vertices.shape)
print("Faces:", faces.shape)

Mesh loaded.
Vertices: (192, 3)
Faces: (124, 3)


In [34]:
#  DETECT GEOMETRIC SEAMS
# (Edges shared by only 1 face - boundary seam)
# (Edges shared by more than 2 faces - non-manifold seam)

edges = mesh.edges_sorted
edges_unique, edges_count = np.unique(edges, axis=0, return_counts=True)

seam_edges = edges_unique[edges_count != 2]   # boundary or abnormal edges
seam_vertices = np.unique(seam_edges)         # vertices involved in seams

print("\nDetected seam vertices:", seam_vertices[:10])
print("Total seam vertices:", len(seam_vertices))



Detected seam vertices: [0 1 2 3 4 5 6 7 8 9]
Total seam vertices: 192


In [35]:
#  ENCODING SEAMS INTO TOKENS

def encode_seams(seam_vertices, vertices):
    tokens = []

    for v in seam_vertices:
        x, y, z = vertices[v]

        tokens.append(("SEAM_VERTEX", int(v)))
        tokens.append(("CORDI", round(float(x), 4), round(float(y), 4), round(float(z), 4)))
        tokens.append(("Done",))

    return tokens


tokens = encode_seams(seam_vertices, vertices)

print("\nSample Tokens:")
print(tokens[:5])



Sample Tokens:
[('SEAM_VERTEX', 0), ('CORDI', 0.0, -1.0, -1.0), ('Done',), ('SEAM_VERTEX', 1), ('CORDI', 0.0, -1.0, -1.0)]


In [36]:
#  DECODER – Rebuild seam dictionary from tokens

def decode_seams(tokens):
    decoded = {}
    current_v = None

    for tok in tokens:
        if tok[0] == "SEAM_VERTEX":
            current_v = tok[1]

        elif tok[0] == "CORDI":
            decoded[current_v] = (tok[1], tok[2], tok[3])

        elif tok[0] == "Donee":
            current_v = None

    return decoded


decoded = decode_seams(tokens)

print("\nDecoded seam (first entry):")
first_key = next(iter(decoded))
print("Vertex:", first_key)
print("Position:", decoded[first_key])



Decoded seam (first entry):
Vertex: 0
Position: (0.0, -1.0, -1.0)
